<a href="https://colab.research.google.com/github/revirevy/notebooks/blob/master/notebooks/HYPIR_4K_image_upscale_by_AIQUEST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎙️ HYPIR - Image Restoration & Upscaling</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Google Colab Free Tier - Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>Harnessing Diffusion-Yielded Score Priors for Image Restoration</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-T4%20GPU-4285F4?style=for-the-badge&logo=google-colab&logoColor=white" />
  <br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

---

### Features
1. **Single Forward Pass Restoration**: Achieves high-fidelity restoration and upscaling using pre-trained score priors without iterative diffusion steps.
2. **Adjustable Texture Richness & Fidelity**: Fully compatible with text-guided restoration and resolution upscaling.
3. **T4 VRAM Optimized**: Operates efficiently within free-tier GPU resources.
4. **Gradio WebUI**: Simple, intuitive interactive interface with standard buttons (Generate, Stop, Clear).

---

### Quick Start
1. Ensure **GPU runtime** is active: **Runtime → Change runtime type → T4 GPU**
2. Run the environment setup cell to clone HYPIR, install packages, and download weights.
3. Launch the Gradio Web Interface cell and click the generated public URL!


In [1]:
#@title 📦 Step 1: Install Dependencies & Setup Environment
# Clones the repository, checks and installs missing dependencies, and downloads model weights.
import os
import sys
import subprocess
import torch

print('=== Colab T4 Environment Setup ===')
try:
    subprocess.run(['nvidia-smi'], check=True)
    print(f'GPU Active: {torch.cuda.get_device_name(0)}')
except Exception:
    print('WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU')

if not os.path.exists('HYPIR'):
    print('Cloning HYPIR repository...')
    subprocess.run(['git', 'clone', 'https://github.com/XPixelGroup/HYPIR.git'], check=True)
else:
    print('HYPIR repository already cloned.')

# Move to the HYPIR directory if not already inside it
if not os.getcwd().endswith('HYPIR'):
    if os.path.exists('HYPIR'):
        os.chdir('HYPIR')
    else:
        print("Warning: HYPIR folder not found and current directory is not HYPIR.")
sys.path.append(os.getcwd())

# Define packages to check and install (excluding torch and torchvision to prevent Colab binary conflicts)
packages_to_check = {
    'omegaconf': 'omegaconf',
    'python-dotenv': 'dotenv',
    'accelerate': 'accelerate',
    'diffusers': 'diffusers',
    'lpips': 'lpips',
    'open_clip_torch': 'open_clip',
    'peft': 'peft',
    'einops': 'einops',
    'pydantic': 'pydantic',
    'opencv-python-headless': 'cv2',
    'timm': 'timm',
    'vision-aided-loss': 'vision_aided_loss',
    'polars': 'polars',
    'tenacity': 'tenacity',
    'openai': 'openai',
    'nest_asyncio': 'nest_asyncio'
}

missing_packages = []
for pkg, import_name in packages_to_check.items():
    try:
        __import__(import_name)
    except ImportError:
        missing_packages.append(pkg)

# Check and upgrade gradio to 6.0.0+ to prevent "HfFolder" ImportError with newer huggingface_hub versions
try:
    import gradio as gr
    parts = [int(x) for x in gr.__version__.split('.') if x.isdigit()]
    if len(parts) >= 1 and parts[0] < 6:
        print(f"Gradio version {gr.__version__} is too old. Upgrading to gradio>=6.0.0...")
        missing_packages.append("gradio>=6.0.0")
except Exception as e:
    print(f"Gradio import failed: {e}. Installing gradio>=6.0.0...")
    missing_packages.append("gradio>=6.0.0")

# Check if torchao needs to be installed or upgraded (minimum 0.16.0 required by PEFT)
try:
    import torchao
    parts = [int(x) for x in torchao.__version__.split('.') if x.isdigit()]
    if len(parts) >= 2 and (parts[0] < 0 or (parts[0] == 0 and parts[1] < 16)):
        print(f"torchao version {torchao.__version__} is too old. Adding to upgrade list...")
        missing_packages.append("torchao>=0.16.0")
except (ImportError, ValueError):
    missing_packages.append("torchao>=0.16.0")

is_gradio_reinstall = False
if missing_packages:
    # If gradio needs to be installed/upgraded, let's force-uninstall first to avoid conflict files
    if any("gradio" in pkg for pkg in missing_packages):
        is_gradio_reinstall = True
        print("Mismatched Gradio installation files detected. Uninstalling existing Gradio package first...")
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'gradio', 'gradio-client'])

    print(f'Installing missing dependencies: {missing_packages}...')
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-cache-dir'] + missing_packages
    print(f"Running: {' '.join(cmd)}")
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    while True:
        output = process.stdout.readline()
        if output == '' and process.poll() is not None:
            break
        if output:
            print(output.strip())
            sys.stdout.flush()
    rc = process.poll()
    if rc != 0:
        raise RuntimeError(f"pip install failed with return code {rc}")
else:
    print('✅ All required dependencies are already installed.')

if not os.path.exists('HYPIR_sd2.pth'):
    print('Downloading HYPIR_sd2.pth pre-trained weights...')
    subprocess.run(['wget', 'https://huggingface.co/lxq007/HYPIR/resolve/main/HYPIR_sd2.pth'], check=True)
else:
    print('HYPIR_sd2.pth already downloaded.')

if is_gradio_reinstall:
    print("\n⚠️ GRADIO WAS REINSTALLED/UPGRADED! ⚠️")
    print("To apply changes and prevent caching/import conflicts, please manually restart your Colab session")
    print("(Runtime -> Restart session) and then re-run the cells.")

print('✅ Setup finished successfully!')

=== Colab T4 Environment Setup ===
GPU Active: Tesla T4
Cloning HYPIR repository...
torchao version 0.10.0 is too old. Adding to upgrade list...
Installing missing dependencies: ['lpips', 'open_clip_torch', 'vision-aided-loss', 'torchao>=0.16.0']...
Running: /usr/bin/python3 -m pip install --no-cache-dir lpips open_clip_torch vision-aided-loss torchao>=0.16.0
Installing build dependencies: started
Installing build dependencies: finished with status 'done'
Getting requirements to build wheel: started
Getting requirements to build wheel: finished with status 'done'
Preparing metadata (pyproject.toml): started
Preparing metadata (pyproject.toml): finished with status 'done'
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 157.0 MB/s eta 0:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 282.0 MB/s eta 0:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 311.8 MB/s eta 0:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 174.5 MB/s eta 0:00:00
Created wheel fo

In [ ]:
#@title 🚀 Step 2: Launch Gradio Web Interface
# Ensure we are in the HYPIR directory if running directly
import os
import sys
if not os.getcwd().endswith('HYPIR') and os.path.exists('HYPIR'):
    os.chdir('HYPIR')
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

import random
import asyncio
import nest_asyncio
nest_asyncio.apply()

# Fix nest_asyncio compatibility with Python 3.12+ (which calls asyncio.run with loop_factory)
import threading
import asyncio.runners
true_original_run = asyncio.runners.run
nest_asyncio.apply()
patched_asyncio_run = asyncio.run

def _patched_run(main, *, debug=False, **kwargs):
    if threading.current_thread() is not threading.main_thread():
        return true_original_run(main, debug=debug, **kwargs)
    return patched_asyncio_run(main, debug=debug)

asyncio.run = _patched_run

import os
import sys

# Mock HfFolder before importing gradio to support older Gradio versions on newer huggingface_hub
try:
    import huggingface_hub.utils
    if not hasattr(huggingface_hub.utils, 'HfFolder'):
        class DummyHfFolder:
            @staticmethod
            def get_token():
                return None
            @staticmethod
            def save_token(token):
                pass
            @staticmethod
            def delete_token():
                pass
        huggingface_hub.utils.HfFolder = DummyHfFolder
except ImportError:
    pass

import gc
import torch
import gradio as gr
import torchvision.transforms as transforms
from accelerate.utils import set_seed
from PIL import Image
import tempfile
import urllib.parse
import uuid

def create_comparison_slider(before_path, after_path):
    if not before_path or not after_path:
        return """
        <div style="text-align: center; padding: 40px; border: 2px dashed #ccc; border-radius: 12px; color: #666; font-family: sans-serif;">
            🔍 Run restoration to view the interactive before vs after comparison slider.
        </div>
        """

    slider_id = f"slider-{uuid.uuid4().hex[:8]}"

    try:
        is_gradio_v4 = gr.__version__.startswith('4.')
    except Exception:
        is_gradio_v4 = False

    prefix = "/file=" if is_gradio_v4 else "/gradio_api/file="

    before_url = f"{prefix}{urllib.parse.quote(before_path)}"
    after_url = f"{prefix}{urllib.parse.quote(after_path)}"

    html = f"""
    <div id="{slider_id}" class="comparison-container" style="position: relative; width: 100%; max-width: 800px; height: 500px; margin: 0 auto; overflow: hidden; border-radius: 12px; box-shadow: 0 8px 30px rgba(0,0,0,0.3); background: #111; user-select: none; -webkit-user-select: none; touch-action: none;">
      <!-- Background Image: Restored -->
      <img src="{after_url}" style="position: absolute; top: 0; left: 0; width: 100%; height: 100%; object-fit: contain; pointer-events: none; user-select: none; -webkit-user-select: none;">
      <span style="position: absolute; bottom: 16px; right: 16px; background: rgba(0,0,0,0.8); color: #fff; padding: 6px 14px; border-radius: 6px; font-size: 13px; font-weight: 600; font-family: sans-serif; letter-spacing: 0.5px; z-index: 4; user-select: none; pointer-events: none;">Restored</span>

      <!-- Foreground Image: Original (Clipped) -->
      <img class="clip-foreground" src="{before_url}" style="position: absolute; top: 0; left: 0; width: 100%; height: 100%; object-fit: contain; pointer-events: none; clip-path: inset(0 50% 0 0); z-index: 2; user-select: none; -webkit-user-select: none;">
      <span style="position: absolute; bottom: 16px; left: 16px; background: rgba(0,0,0,0.8); color: #fff; padding: 6px 14px; border-radius: 6px; font-size: 13px; font-weight: 600; font-family: sans-serif; letter-spacing: 0.5px; z-index: 4; user-select: none; pointer-events: none;">Original</span>

      <!-- Slider Bar and Drag Handle -->
      <div class="clip-slider-handle" style="position: absolute; top: 0; bottom: 0; left: 50%; width: 4px; background: #667eea; cursor: ew-resize; z-index: 5; transform: translateX(-50%); display: flex; align-items: center; justify-content: center; touch-action: none;">
        <div style="width: 38px; height: 38px; background: #fff; border: 3px solid #667eea; border-radius: 50%; display: flex; align-items: center; justify-content: center; box-shadow: 0 4px 10px rgba(0,0,0,0.3); color: #667eea; font-weight: bold; font-size: 18px; transition: transform 0.1s; user-select: none; -webkit-user-select: none; touch-action: none;">↔</div>
      </div>
    </div>
    """
    return html

# Ensure the module is in sys.path
sys.path.append(os.getcwd())

try:
    from HYPIR.enhancer.sd2 import SD2Enhancer
except ModuleNotFoundError:
    from enhancer.sd2 import SD2Enhancer

# Close existing Gradio instances to free ports/VRAM
try:
    gr.close_all()
    print("Closed existing Gradio instances.")
except Exception:
    pass

# Check and open default error image or create fallback
error_image_path = os.path.join("assets", "gradio_error_img.png")
if os.path.exists(error_image_path):
    error_image = Image.open(error_image_path)
else:
    # Fallback solid color error image if file not found
    error_image = Image.new("RGB", (512, 512), (180, 50, 50))

# Limit max output resolution to avoid VRAM exhaustion on free T4
max_size = (4096, 4096)
to_tensor = transforms.ToTensor()

print("Initializing HYPIR SD2Enhancer model...")
model = SD2Enhancer(
    base_model_path="sd2-community/stable-diffusion-2-1-base",
    weight_path="HYPIR_sd2.pth",
    lora_modules=[
        "to_k",
        "to_q",
        "to_v",
        "to_out.0",
        "conv",
        "conv1",
        "conv2",
        "conv_shortcut",
        "conv_out",
        "proj_in",
        "proj_out",
        "ff.net.2",
        "ff.net.0.proj",
    ],
    lora_rank=256,
    model_t=200,
    coeff_t=200,
    device="cuda",
)
model.init_models()
print("✅ Model loaded successfully on GPU!")

def process(image, prompt, upscale, seed, progress=gr.Progress(track_tqdm=True)):
    if image is None:
        return None, None, "⚠️ Please upload an image."

    if seed == -1:
        seed = random.randint(0, 2**32 - 1)
    set_seed(seed)

    try:
        progress(0.1, desc="Preparing image...")
        img_rgb = image.convert("RGB")
        orig_w, orig_h = img_rgb.size

        # Calculate target upscaled dimensions
        out_w, out_h = int(orig_w * upscale), int(orig_h * upscale)

        # Enforce strict 4k output limits
        max_output_w, max_output_h = max_size
        if out_w > max_output_w or out_h > max_output_h:
            max_input_side = max_output_w // upscale
            # Ensure max_input_side is multiple of 16 for user convenience
            max_input_side = (max_input_side // 16) * 16
            return error_image, None, (
                f"❌ Error: The requested upscaled resolution of {out_w}x{out_h} exceeds the 4k limit ({max_output_w}x{max_output_h}).\n"
                f"Please manually resize your input image to at most {max_input_side}x{max_input_side} "
                f"before uploading to run successfully with a {upscale}x upscale factor."
            )

        progress(0.3, desc="Restoring / Upscaling image...")
        image_tensor = to_tensor(img_rgb).unsqueeze(0)

        with torch.no_grad():
            enhanced_pil = model.enhance(
                lq=image_tensor,
                prompt=prompt,
                upscale=upscale,
                return_type="pil",
            )[0]

        progress(0.9, desc="Finalizing output formatting...")
        gc.collect()
        torch.cuda.empty_cache()

        # Save output images as JPG inside a temp directory to enforce JPG downloads.
        # Disable chroma subsampling (subsampling=0) and use 100% quality to avoid any compression artifacts.
        out_dir = tempfile.mkdtemp()
        output_path = os.path.join(out_dir, "restored_output.jpg")
        enhanced_pil.save(output_path, format="JPEG", quality=100, subsampling=0)

        input_temp_path = os.path.join(out_dir, "original_input.jpg")
        img_rgb.save(input_temp_path, format="JPEG", quality=100, subsampling=0)

        progress(1.0, desc="Done!")
        slider_html = create_comparison_slider(input_temp_path, output_path)
        return output_path, slider_html, f"Success! :)\nUsed Prompt: {prompt}\nUsed Seed: {seed}"

    except Exception as e:
        gc.collect()
        torch.cuda.empty_cache()
        # Save error image as jpg if it fails
        try:
            out_dir = tempfile.mkdtemp()
            err_path = os.path.join(out_dir, "error.jpg")
            error_image.save(err_path, format="JPEG", quality=100, subsampling=0)
            return err_path, create_comparison_slider(None, None), f"Failed: {str(e)} :("
        except Exception:
            return None, create_comparison_slider(None, None), f"Failed: {str(e)} :("

# Gradio Custom Styling CSS
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.social-buttons { display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }
.social-btn { padding: 10px 24px; border-radius: 8px; font-weight: 700; font-size: 15px; text-decoration: none; display: inline-block; color: white; transition: all 0.3s; box-shadow: 0 4px 12px rgba(0,0,0,0.2); }
.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }
.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }
.x-btn { background: linear-gradient(135deg, #000000 0%, #333333 100%); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; }
"""

SLIDER_JS = """
() => {
    setTimeout(() => {
        const container = document.querySelector('.comparison-container');
        if (!container) return;
        if (container.dataset.initialized) return;
        container.dataset.initialized = "true";

        const foreground = container.querySelector('.clip-foreground');
        const handle = container.querySelector('.clip-slider-handle');
        const handleIcon = handle.querySelector('div');

        let isDragging = false;

        function setSliderPosition(clientX) {
            const rect = container.getBoundingClientRect();
            let pos = (clientX - rect.left) / rect.width;
            if (pos < 0) pos = 0;
            if (pos > 1) pos = 1;

            const pct = pos * 100;
            foreground.style.clipPath = `inset(0 ` + (100 - pct) + `% 0 0)`;
            handle.style.left = pct + "%";
        }

        function onPointerDown(e) {
            isDragging = true;
            handleIcon.style.transform = 'scale(1.15)';
            setSliderPosition(e.clientX);
            try {
                handle.setPointerCapture(e.pointerId);
            } catch(err) {}
            e.preventDefault();
        }

        function onPointerMove(e) {
            if (!isDragging) return;
            setSliderPosition(e.clientX);
        }

        function onPointerUp(e) {
            if (isDragging) {
                isDragging = false;
                handleIcon.style.transform = 'scale(1)';
                try {
                    handle.releasePointerCapture(e.pointerId);
                } catch(err) {}
            }
        }

        handle.addEventListener('pointerdown', onPointerDown);
        handle.addEventListener('pointermove', onPointerMove);
        handle.addEventListener('pointerup', onPointerUp);
        handle.addEventListener('pointercancel', onPointerUp);

        container.addEventListener('pointerdown', function(e) {
            if (e.target !== handle && !handle.contains(e.target)) {
                setSliderPosition(e.clientX);
                isDragging = true;
                handleIcon.style.transform = 'scale(1.15)';
                try {
                    handle.setPointerCapture(e.pointerId);
                } catch(err) {}
            }
        });
    }, 150);
}
"""

with gr.Blocks(title="HYPIR Image Restoration | AIQUEST Academy") as demo:
    gr.HTML('<div class="brand-header"><div class="brand-title">🎙️ HYPIR - Image Restoration & Upscaling</div><div class="brand-subtitle">Google Colab Free Tier - Created by <strong>AIQUEST Academy</strong></div><div class="social-buttons"><a href="https://youtube.com/@aiquestacademy" target="_blank" class="social-btn youtube-btn">▶️ Subscribe on YouTube</a><a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a></div></div>')

    gr.Markdown(
        "Upload an image, type an optional descriptive prompt to guide texture and detail generation, "
        "and click **✨ Restore & Upscale** to process the image in a single pass."
    )

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(label="Input Image", type="pil")
            prompt_input = gr.Textbox(label="Prompt (Guides details generation)", placeholder="e.g. a high quality face photo, sharp details...")
            upscale_slider = gr.Slider(minimum=1, maximum=8, step=1, value=1, label="Upscale Factor")
            seed_input = gr.Number(label="Seed (-1 for random)", value=-1)

            with gr.Row():
                gen_btn = gr.Button("✨ Restore & Upscale", variant="primary", size="lg", elem_id="gen-btn")
                stop_btn = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")

        with gr.Column():
            output_image = gr.Image(label="Restored Output", type="filepath")
            status_output = gr.Textbox(label="Status / Log", interactive=False)

    with gr.Row():
        with gr.Column():
            comparison_slider = gr.HTML(value=create_comparison_slider(None, None))

    gen_event = gen_btn.click(
        fn=process,
        inputs=[input_image, prompt_input, upscale_slider, seed_input],
        outputs=[output_image, comparison_slider, status_output]
    )
    gen_event.then(fn=None, js=SLIDER_JS)

    stop_btn.click(fn=None, cancels=[gen_event])
    clear_btn.click(
        fn=lambda: (None, "", 1, -1, None, create_comparison_slider(None, None), ""),
        outputs=[input_image, prompt_input, upscale_slider, seed_input, output_image, comparison_slider, status_output]
    )

print("Launching Gradio interface...")
demo.queue()
demo.launch(share=True, inline=False, debug=True, css=CSS, theme=gr.themes.Soft(), allowed_paths=[tempfile.gettempdir(), os.getcwd()])

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Closed existing Gradio instances.
Initializing HYPIR SD2Enhancer model...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


scheduler_config.json:   0%|          | 0.00/346 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/807 [00:00<?, ?B/s]

...

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/911 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B / 3.46GB            

unet/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

Load model weights from HYPIR_sd2.pth
✅ Model loaded successfully on GPU!
Launching Gradio interface...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c92b27e1420bfa7c8a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
<div align="center">
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>
</div>
<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>
---